# Qwen2-VL prompting on PathEval (Kaggle runner)

Thin runner for `src/prompting.py`: clones the team repo, loads Qwen2-VL-7B once, runs prompting
conditions, and summarizes. All logic lives in the repo; this notebook only calls it.

**Settings:** Accelerator = **GPU T4 x2**, Internet = On.
**Inputs:** add the `patheval-2d-test` dataset (Add Input → Your Datasets).
**Secrets (Add-ons → Secrets):** `GITHUB_TOKEN`, a classic GitHub token with `repo` scope (the team repo is private).

## 1. Settings

In [ ]:
REPO = "zoekrishtul/Deep-Learning-Final-Project"
BRANCH = "gaurav/prompting"        # change to "main" once merged
N_PER_SCENARIO = 10                # 10 -> 150 samples (~5 min/condition); None -> all 1050
BATCH_SIZE = 8
OUT_DIR = "/kaggle/working/results"

## 2. Get the code
Cloned to `/tmp` (not `/kaggle/working`) so the token never ends up in saved notebook output.

In [ ]:
import os, sys, subprocess
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None
    print("No GITHUB_TOKEN secret; cloning will only work if the repo is public")

auth = f"{token}@" if token else ""
subprocess.run(["rm", "-rf", "/tmp/repo"], check=True)
subprocess.run(["git", "clone", "-q", "--branch", BRANCH,
                f"https://{auth}github.com/{REPO}.git", "/tmp/repo"], check=True)
subprocess.run(["git", "-C", "/tmp/repo", "remote", "set-url", "origin",
                f"https://github.com/{REPO}.git"], check=True)   # drop the token from git config
os.chdir("/tmp/repo")
sys.path.insert(0, "/tmp/repo")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Find the data
Uses `test_2D.jsonl` if the dataset has it; otherwise converts the older `test_2D_clean.json`.

In [ ]:
import glob, json
from src.data import load_manifest, manifest_from_clean, write_jsonl

found = glob.glob("/kaggle/input/**/test_2D.jsonl", recursive=True)
if found:
    MANIFEST = found[0]
else:
    clean = glob.glob("/kaggle/input/**/test_2D_clean.json", recursive=True)[0]
    MANIFEST = "/kaggle/working/test_2D.jsonl"
    write_jsonl(manifest_from_clean(json.load(open(clean))), MANIFEST)
DATA_ROOT = os.path.dirname(glob.glob("/kaggle/input/**/test_2D*.json*", recursive=True)[0])
rows = load_manifest(MANIFEST)
print(len(rows), "samples | manifest:", MANIFEST, "| data root:", DATA_ROOT)

## 4. Load Qwen2-VL-7B (float16, split across both T4s)

In [ ]:
from src.prompting import QwenPathEvaluator, run, summarize, print_summary
evaluator = QwenPathEvaluator()
evaluator.describe()

## 5. Direct prompting: original and swapped images

In [ ]:
direct = run(MANIFEST, DATA_ROOT, f"{OUT_DIR}/direct_2D.jsonl", evaluator,
             strategy="direct", n_per_scenario=N_PER_SCENARIO, batch_size=BATCH_SIZE)
direct_swapped = run(MANIFEST, DATA_ROOT, f"{OUT_DIR}/direct_2D_swapped.jsonl", evaluator,
                     strategy="direct", swap=True, n_per_scenario=N_PER_SCENARIO,
                     batch_size=BATCH_SIZE)

## 6. Quick summary (official metrics: `src/evaluation.py`)

In [ ]:
print_summary(summarize(direct, swapped_path=direct_swapped, per_scenario=True))

Download `/kaggle/working/results/` from the Output panel before stopping the session
(results are not committed to Git; see `results/README.md`).